# Built-In Model Monitoring with Snowflake

Snowflake's Model Monitor is a managed service that automatically tracks drift, performance, and statistical metrics for models registered in the Model Registry. You create a monitor, point it at a predictions table, and Snowflake handles the rest -- computing metrics on a schedule and exposing them via SQL functions and the Snowsight UI.

This notebook walks through the full Model Monitor lifecycle:

1. **Create a prediction source table** -- where your model's predictions live
2. **Create a monitor (no baseline)** -- get statistical metrics immediately
3. **Set a baseline** -- unlock drift metrics (PSI, Jensen-Shannon, Wasserstein)
4. **Add ground truth labels** -- unlock performance metrics (accuracy, F1, precision, recall)
5. **Segmented monitoring** -- track metrics per data subgroup
6. **Monitor management** -- suspend, resume, inspect, and configure

### What You Get Out of the Box

| Category | Metrics | Requires |
|----------|---------|----------|
| **Statistical** | COUNT, NULL_COUNT, MEAN, STDDEV, MIN, MAX | Source table only |
| **Drift** | POPULATION_STABILITY_INDEX, JENSEN_SHANNON, WASSERSTEIN, DIFFERENCE_OF_MEANS | Baseline table |
| **Performance (classification)** | CLASSIFICATION_ACCURACY, PRECISION, RECALL, F1_SCORE | Ground truth labels |
| **Performance (regression)** | MAE, RMSE, MAPE | Ground truth labels |

### Prerequisites

Run `00_setup.ipynb` first to create the infrastructure, data, and register `CHURN_MODEL` V1/V2.

## 1. Setup and Connect

Import libraries and establish a Snowflake session. We retrieve the registered CHURN_MODEL V1 from the Model Registry and load the test data for generating predictions.

**Snowsight (Workspaces):** The session is pre-authenticated -- no configuration needed.

**Local development:** Falls back to keypair authentication. Update the `LOCAL CONFIG` section below.

In [ ]:
import numpy as np
import pandas as pd
from snowflake.snowpark import Session
from snowflake.ml.registry import Registry

In [ ]:
try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
except Exception:
    from cryptography.hazmat.primitives import serialization
    from pathlib import Path

    # ── LOCAL CONFIG (update these for your environment) ──
    ACCOUNT = "<your-account-identifier>"
    USER = "<your-username>"
    ROLE = "ACCOUNTADMIN"
    KEY_PATH = Path.home() / ".snowflake" / "keys" / "rsa_key.p8"
    # ──────────────────────────────────────────────────────

    with open(KEY_PATH, "rb") as f:
        private_key = serialization.load_pem_private_key(f.read(), password=None)

    private_key_bytes = private_key.private_bytes(
        encoding=serialization.Encoding.DER,
        format=serialization.PrivateFormat.PKCS8,
        encryption_algorithm=serialization.NoEncryption()
    )

    session = Session.builder.configs({
        "account": ACCOUNT,
        "user": USER,
        "private_key": private_key_bytes,
        "role": ROLE,
        "database": "ML_DEMO",
        "schema": "ML_CHURN",
        "warehouse": "ML_CHURN_WH"
    }).create()

print(f"Connected as: {session.get_current_role()}")
print(f"Database: {session.get_current_database()}")
print(f"Schema: {session.get_current_schema()}")

In [ ]:
FEATURE_COLS = ["TENURE_MONTHS", "MONTHLY_CHARGES", "TOTAL_CHARGES",
                "CONTRACT_TYPE", "NUM_SUPPORT_TICKETS", "INTERNET_SERVICE"]

# Load test data and retrieve model
test_data = session.table("ML_DEMO.ML_CHURN.CHURN_TEST").to_pandas()
train_data = session.table("ML_DEMO.ML_CHURN.CHURN_TRAIN").to_pandas()

reg = Registry(session=session)
model = reg.get_model("CHURN_MODEL")
mv_v1 = model.version("V1")

print(f"Test data: {len(test_data)} rows")
print(f"Train data: {len(train_data)} rows")
print(f"Model: CHURN_MODEL V1")

## 2. Create a Prediction Source Table

The Model Monitor reads predictions from a **source table** -- a Snowflake table that contains your model's outputs along with a timestamp column. In production, your inference pipeline populates this table continuously (via batch inference, a streaming service, or an application).

For this demo, we generate predictions using `mv.run()` (batch inference on the warehouse) and spread timestamps across the last 24 hours so the monitor has time-series data to aggregate.

The source table includes:
- **Input features** (the model's input columns)
- **Prediction output** (the model's predicted score/class)
- **PREDICTION_TIMESTAMP** (`TIMESTAMP_NTZ`) -- when the prediction was made
- **ACTUAL_CHURNED** -- ground truth labels (used later for performance metrics)
- **CONTRACT_TYPE_SEG** -- a STRING version of CONTRACT_TYPE for segmented monitoring

In [ ]:
# Generate predictions using batch inference
test_snowdf = session.create_dataframe(test_data[FEATURE_COLS])
preds_snowdf = mv_v1.run(test_snowdf, function_name="predict")
preds_df = preds_snowdf.to_pandas()

# The prediction output column name varies -- find it
pred_col = [c for c in preds_df.columns if c not in FEATURE_COLS][0]
print(f"Prediction column: {pred_col}")
preds_df.head()

In [ ]:
# Build the source table with timestamps, actuals, and segment column
source_df = preds_df.copy()

# Spread timestamps across last 24 hours so aggregation windows have data
source_df["PREDICTION_TIMESTAMP"] = pd.date_range(
    end=pd.Timestamp.now().floor("s"), periods=len(source_df), freq="86s"
)

# Add ground truth labels (we'll use these in section 6)
source_df["ACTUAL_CHURNED"] = test_data["CHURNED"].values

# Add string segment column (Model Monitor requires STRING type for segments)
source_df["CONTRACT_TYPE_SEG"] = test_data["CONTRACT_TYPE"].astype(str).values

# Rename prediction column to something readable
source_df = source_df.rename(columns={pred_col: "PREDICTED_CHURNED"})

# Convert timestamp to string for reliable Snowflake ingestion, then cast server-side
source_df["PREDICTION_TIMESTAMP"] = source_df["PREDICTION_TIMESTAMP"].dt.strftime("%Y-%m-%d %H:%M:%S")

# Write with auto_create (creates as VARCHAR), then replace with proper typed table
session.write_pandas(source_df, table_name="CHURN_PREDICTIONS_STAGING", database="ML_DEMO",
                     schema="ML_CHURN", overwrite=True, auto_create_table=True)

session.sql("DROP TABLE IF EXISTS ML_DEMO.ML_CHURN.CHURN_PREDICTIONS").collect()
session.sql("""
CREATE TABLE ML_DEMO.ML_CHURN.CHURN_PREDICTIONS AS
SELECT
    TENURE_MONTHS,
    MONTHLY_CHARGES,
    TOTAL_CHARGES,
    CONTRACT_TYPE,
    NUM_SUPPORT_TICKETS,
    INTERNET_SERVICE,
    PREDICTED_CHURNED,
    PREDICTION_TIMESTAMP::TIMESTAMP_NTZ AS PREDICTION_TIMESTAMP,
    ACTUAL_CHURNED,
    CONTRACT_TYPE_SEG
FROM ML_DEMO.ML_CHURN.CHURN_PREDICTIONS_STAGING
""").collect()
session.sql("DROP TABLE IF EXISTS ML_DEMO.ML_CHURN.CHURN_PREDICTIONS_STAGING").collect()

# Verify column types
col_check = session.sql("SELECT PREDICTION_TIMESTAMP FROM ML_DEMO.ML_CHURN.CHURN_PREDICTIONS LIMIT 1").to_pandas()
print(f"Created CHURN_PREDICTIONS: {len(source_df)} rows")
print(f"PREDICTION_TIMESTAMP type check: {col_check.dtypes['PREDICTION_TIMESTAMP']}")
source_df[["PREDICTED_CHURNED", "ACTUAL_CHURNED", "PREDICTION_TIMESTAMP", "CONTRACT_TYPE_SEG"]].head()

## 3. Create a Monitor -- Statistical Metrics Only

We start with the simplest monitor configuration: just a model, a source table, and a timestamp column. This gives us **statistical metrics** (count, mean, stddev, min, max, null count) immediately -- no baseline or ground truth needed.

**Key parameters:**
- `MODEL` / `VERSION`: The registered model and version to monitor
- `SOURCE`: The predictions table
- `TIMESTAMP_COLUMN`: Must be `TIMESTAMP_NTZ` -- used to bucket metrics into aggregation windows
- `PREDICTION_SCORE_COLUMNS`: The model's output column (use `PREDICTION_CLASS_COLUMNS` for categorical outputs)
- `AGGREGATION_WINDOW`: How wide each time bucket is (must be in days, e.g., `'1 day'`)
- `REFRESH_INTERVAL`: How often the monitor recomputes metrics (minimum `'60 seconds'`)
- `WAREHOUSE`: Compute for the monitor's background refresh job

In [ ]:
# Drop any existing monitor from previous runs
session.sql("DROP MODEL MONITOR IF EXISTS ML_DEMO.ML_CHURN.CHURN_MONITOR").collect()

# For binary classification, use PREDICTION_CLASS_COLUMNS (not SCORE)
# Use PREDICTION_SCORE_COLUMNS for regression models instead
session.sql("""
CREATE MODEL MONITOR ML_DEMO.ML_CHURN.CHURN_MONITOR WITH
    MODEL = ML_DEMO.ML_CHURN.CHURN_MODEL
    VERSION = 'V1'
    FUNCTION = 'predict'
    SOURCE = ML_DEMO.ML_CHURN.CHURN_PREDICTIONS
    WAREHOUSE = ML_CHURN_WH
    REFRESH_INTERVAL = '1 minute'
    AGGREGATION_WINDOW = '1 day'
    TIMESTAMP_COLUMN = PREDICTION_TIMESTAMP
    PREDICTION_CLASS_COLUMNS = ('PREDICTED_CHURNED')
""").collect()

print("Created model monitor CHURN_MONITOR")

In [ ]:
# Verify the monitor was created
monitor_list = session.sql("SHOW MODEL MONITORS LIKE 'CHURN_MONITOR' IN SCHEMA ML_DEMO.ML_CHURN").to_pandas()
print("Monitor status:")
monitor_list[["name", "monitor_state"]].head() if "monitor_state" in monitor_list.columns else monitor_list.head()

### Query Statistical Metrics

Statistical metrics are available immediately -- they describe the distribution of your prediction column over time. These work without any baseline or ground truth.

Use the `MODEL_MONITOR_STAT_METRIC()` table function to query them. The `GRANULARITY` parameter controls the time bucketing.

In [ ]:
# Query COUNT metric -- how many predictions per aggregation window
import time
print("Waiting 60s for initial monitor refresh...")
time.sleep(60)

try:
    count_metrics = session.sql("""
    SELECT * FROM TABLE(
        MODEL_MONITOR_STAT_METRIC(
            'ML_DEMO.ML_CHURN.CHURN_MONITOR',
            METRIC_NAME => 'COUNT',
            COLUMN_NAME => '"PREDICTED_CHURNED"',
            GRANULARITY => '1 DAY'
        )
    )
    """).to_pandas()
    print("COUNT metric (predictions per day):")
    count_metrics
except Exception as e:
    print(f"Metrics not yet available (monitor may still be initializing): {e}")

In [ ]:
# Query MEAN and STDDEV -- distribution summary of predictions
try:
    mean_metrics = session.sql("""
    SELECT * FROM TABLE(
        MODEL_MONITOR_STAT_METRIC(
            'ML_DEMO.ML_CHURN.CHURN_MONITOR',
            METRIC_NAME => 'MEAN',
            COLUMN_NAME => '"PREDICTED_CHURNED"',
            GRANULARITY => '1 DAY'
        )
    )
    """).to_pandas()
    print("MEAN of predictions per day:")
    display(mean_metrics)
except Exception as e:
    print(f"Not yet available: {e}")

try:
    stddev_metrics = session.sql("""
    SELECT * FROM TABLE(
        MODEL_MONITOR_STAT_METRIC(
            'ML_DEMO.ML_CHURN.CHURN_MONITOR',
            METRIC_NAME => 'STDDEV',
            COLUMN_NAME => '"PREDICTED_CHURNED"',
            GRANULARITY => '1 DAY'
        )
    )
    """).to_pandas()
    print("\nSTDDEV of predictions per day:")
    display(stddev_metrics)
except Exception as e:
    print(f"Not yet available: {e}")

## 4. Set a Baseline for Drift Detection

**Drift metrics compare the current prediction distribution against a reference (baseline) distribution.** Without a baseline, you only get statistical metrics -- with one, you unlock four drift metrics:

- **POPULATION_STABILITY_INDEX**: Measures how much the prediction distribution has shifted. PSI < 0.1 = no significant drift, 0.1-0.25 = moderate, > 0.25 = significant.
- **JENSEN_SHANNON**: Symmetric measure of distributional distance (0 = identical, 1 = maximally different).
- **WASSERSTEIN**: Earth mover's distance between distributions.
- **DIFFERENCE_OF_MEANS**: Simple shift in average predictions.

The baseline table has the same schema as the source table but contains predictions from a known-good period -- typically your training or validation data.

We create a baseline by running the same model on the training data, then attach it with `ALTER MODEL MONITOR ... SET BASELINE`.

In [ ]:
# Create baseline table from training data predictions
train_snowdf = session.create_dataframe(train_data[FEATURE_COLS])
baseline_preds = mv_v1.run(train_snowdf, function_name="predict").to_pandas()

# Find prediction column and rename to match source table
baseline_pred_col = [c for c in baseline_preds.columns if c not in FEATURE_COLS][0]
baseline_df = baseline_preds.rename(columns={baseline_pred_col: "PREDICTED_CHURNED"})

# Add timestamp as string (same pattern as source table)
baseline_ts = (pd.Timestamp.now().floor("s") - pd.Timedelta(days=30)).strftime("%Y-%m-%d %H:%M:%S")
baseline_df["PREDICTION_TIMESTAMP"] = baseline_ts

# Baseline must have ALL columns that the source table has
baseline_df["ACTUAL_CHURNED"] = train_data["CHURNED"].values

# Add segment column
baseline_df["CONTRACT_TYPE_SEG"] = train_data["CONTRACT_TYPE"].astype(str).values

# Write staging then CTAS with proper types
session.write_pandas(baseline_df, table_name="CHURN_BASELINE_STAGING", database="ML_DEMO",
                     schema="ML_CHURN", overwrite=True, auto_create_table=True)

session.sql("DROP TABLE IF EXISTS ML_DEMO.ML_CHURN.CHURN_BASELINE").collect()
session.sql("""
CREATE TABLE ML_DEMO.ML_CHURN.CHURN_BASELINE AS
SELECT
    TENURE_MONTHS,
    MONTHLY_CHARGES,
    TOTAL_CHARGES,
    CONTRACT_TYPE,
    NUM_SUPPORT_TICKETS,
    INTERNET_SERVICE,
    PREDICTED_CHURNED,
    PREDICTION_TIMESTAMP::TIMESTAMP_NTZ AS PREDICTION_TIMESTAMP,
    ACTUAL_CHURNED,
    CONTRACT_TYPE_SEG
FROM ML_DEMO.ML_CHURN.CHURN_BASELINE_STAGING
""").collect()
session.sql("DROP TABLE IF EXISTS ML_DEMO.ML_CHURN.CHURN_BASELINE_STAGING").collect()

print(f"Created CHURN_BASELINE: {len(baseline_df)} rows")

In [ ]:
# Attach the baseline to the existing monitor
session.sql("""
ALTER MODEL MONITOR ML_DEMO.ML_CHURN.CHURN_MONITOR SET
    BASELINE = ML_DEMO.ML_CHURN.CHURN_BASELINE
""").collect()
print("Baseline set on CHURN_MONITOR -- drift metrics now available")

### Query Drift Metrics

Now that a baseline is set, we can query drift metrics using `MODEL_MONITOR_DRIFT_METRIC()`. This compares the current prediction distribution against the baseline for each aggregation window.

**Available drift metrics** -- use the `METRIC_NAME` parameter to choose which one to query:

| Metric Name | What it measures | Interpretation |
|-------------|-----------------|----------------|
| `POPULATION_STABILITY_INDEX` | How much the overall distribution shape has shifted | < 0.1 = stable, 0.1-0.25 = moderate drift, > 0.25 = significant drift |
| `JENSEN_SHANNON` | Symmetric statistical distance between distributions | 0 = identical, 1 = maximally different. Good general-purpose drift metric |
| `WASSERSTEIN` | "Earth mover's distance" -- minimum cost to transform one distribution into another | Scale depends on data; compare across time windows rather than using fixed thresholds |
| `DIFFERENCE_OF_MEANS` | Simple shift in the average prediction value | Positive = predictions trending higher, negative = trending lower. Easy to interpret but misses distributional shape changes |

Below we query PSI and Jensen-Shannon as examples. Replace the `METRIC_NAME` value to try any of the four.

In [ ]:
# Wait for monitor to refresh with baseline data
print("Waiting 60s for monitor refresh with baseline...")
time.sleep(60)

# Query PSI (Population Stability Index)
try:
    psi_metrics = session.sql("""
    SELECT * FROM TABLE(
        MODEL_MONITOR_DRIFT_METRIC(
            'ML_DEMO.ML_CHURN.CHURN_MONITOR',
            METRIC_NAME => 'POPULATION_STABILITY_INDEX',
            COLUMN_NAME => '"PREDICTED_CHURNED"',
            GRANULARITY => '1 DAY'
        )
    )
    """).to_pandas()
    print("Population Stability Index -- current vs baseline:")
    display(psi_metrics)
except Exception as e:
    print(f"PSI not yet available: {e}")

# Query Jensen-Shannon Distance
try:
    js_metrics = session.sql("""
    SELECT * FROM TABLE(
        MODEL_MONITOR_DRIFT_METRIC(
            'ML_DEMO.ML_CHURN.CHURN_MONITOR',
            METRIC_NAME => 'JENSEN_SHANNON',
            COLUMN_NAME => '"PREDICTED_CHURNED"',
            GRANULARITY => '1 DAY'
        )
    )
    """).to_pandas()
    print("\nJensen-Shannon Distance -- current vs baseline:")
    display(js_metrics)
except Exception as e:
    print(f"Jensen-Shannon not yet available: {e}")

## 5. Add Ground Truth for Performance Metrics

**Performance metrics (accuracy, F1, precision, recall) require ground truth labels** -- the actual outcomes that you compare predictions against. In production, these often arrive late (e.g., you predict churn today, but learn the outcome 30 days later).

To enable performance metrics, you must specify `ACTUAL_SCORE_COLUMNS` (or `ACTUAL_CLASS_COLUMNS`) when creating the monitor. This parameter **cannot be added after creation** -- you must drop and recreate the monitor.

We recreate the monitor with the same configuration plus the `ACTUAL_SCORE_COLUMNS` parameter pointing to the `ACTUAL_CHURNED` column in our source table.

In [ ]:
# Must drop and recreate -- ACTUAL columns can't be added to an existing monitor
session.sql("DROP MODEL MONITOR IF EXISTS ML_DEMO.ML_CHURN.CHURN_MONITOR").collect()

# For binary classification: use PREDICTION_CLASS_COLUMNS + ACTUAL_CLASS_COLUMNS
# (PREDICTION_SCORE_COLUMNS + ACTUAL_SCORE_COLUMNS is for regression)
session.sql("""
CREATE MODEL MONITOR ML_DEMO.ML_CHURN.CHURN_MONITOR WITH
    MODEL = ML_DEMO.ML_CHURN.CHURN_MODEL
    VERSION = 'V1'
    FUNCTION = 'predict'
    SOURCE = ML_DEMO.ML_CHURN.CHURN_PREDICTIONS
    WAREHOUSE = ML_CHURN_WH
    REFRESH_INTERVAL = '1 minute'
    AGGREGATION_WINDOW = '1 day'
    TIMESTAMP_COLUMN = PREDICTION_TIMESTAMP
    PREDICTION_CLASS_COLUMNS = ('PREDICTED_CHURNED')
    ACTUAL_CLASS_COLUMNS = ('ACTUAL_CHURNED')
    BASELINE = ML_DEMO.ML_CHURN.CHURN_BASELINE
""").collect()

print("Recreated CHURN_MONITOR with ground truth (ACTUAL_CHURNED) and baseline")

### Query Performance Metrics

With ground truth attached, we now have access to the full suite of classification performance metrics. Use `MODEL_MONITOR_PERFORMANCE_METRIC()` to query them.

For binary classification, available metrics are:
- **ACCURACY**: Fraction of correct predictions
- **PRECISION**: Of predicted positives, how many are actually positive
- **RECALL**: Of actual positives, how many were correctly predicted
- **F1**: Harmonic mean of precision and recall

In [ ]:
# Wait for monitor to refresh with actuals
print("Waiting 60s for monitor refresh with ground truth...")
time.sleep(60)

# Query Accuracy
try:
    accuracy = session.sql("""
    SELECT * FROM TABLE(
        MODEL_MONITOR_PERFORMANCE_METRIC(
            'ML_DEMO.ML_CHURN.CHURN_MONITOR',
            METRIC_NAME => 'CLASSIFICATION_ACCURACY',
            GRANULARITY => '1 DAY'
        )
    )
    """).to_pandas()
    print("Classification Accuracy per day:")
    display(accuracy)
except Exception as e:
    print(f"Accuracy not yet available: {e}")

In [ ]:
# Query F1 Score
try:
    f1 = session.sql("""
    SELECT * FROM TABLE(
        MODEL_MONITOR_PERFORMANCE_METRIC(
            'ML_DEMO.ML_CHURN.CHURN_MONITOR',
            METRIC_NAME => 'F1_SCORE',
            GRANULARITY => '1 DAY'
        )
    )
    """).to_pandas()
    print("F1 Score per day:")
    display(f1)
except Exception as e:
    print(f"F1 not yet available: {e}")

# Query Precision and Recall
try:
    precision = session.sql("""
    SELECT * FROM TABLE(
        MODEL_MONITOR_PERFORMANCE_METRIC(
            'ML_DEMO.ML_CHURN.CHURN_MONITOR',
            METRIC_NAME => 'PRECISION',
            GRANULARITY => '1 DAY'
        )
    )
    """).to_pandas()
    
    recall = session.sql("""
    SELECT * FROM TABLE(
        MODEL_MONITOR_PERFORMANCE_METRIC(
            'ML_DEMO.ML_CHURN.CHURN_MONITOR',
            METRIC_NAME => 'RECALL',
            GRANULARITY => '1 DAY'
        )
    )
    """).to_pandas()
    
    print("\nPrecision per day:")
    display(precision)
    print("\nRecall per day:")
    display(recall)
except Exception as e:
    print(f"Precision/Recall not yet available: {e}")

## 6. Segmented Monitoring

Segmented monitoring lets you track metrics **per data subgroup** -- for example, accuracy by contract type, or drift by geographic region. This is critical for catching issues that are invisible in aggregate metrics (e.g., the model performs well overall but fails on month-to-month customers).

**Requirements for segment columns:**
- Must be `STRING` type (cast integers/categoricals to string)
- Maximum 5 segment columns per monitor
- Recommended < 25 unique values per column
- Segment values are case-sensitive

We add `CONTRACT_TYPE_SEG` as a segment column using `ALTER MODEL MONITOR ... ADD SEGMENT_COLUMN`, then query metrics filtered by segment.

In [ ]:
# Add a segment column to the existing monitor
session.sql("""
ALTER MODEL MONITOR ML_DEMO.ML_CHURN.CHURN_MONITOR
    ADD SEGMENT_COLUMN = 'CONTRACT_TYPE_SEG'
""").collect()
print("Added segment column CONTRACT_TYPE_SEG to monitor")

In [ ]:
# Wait for segment data to be computed
print("Waiting 60s for segmented metrics...")
time.sleep(60)

# Query accuracy for CONTRACT_TYPE = '0' (month-to-month)
try:
    seg_accuracy_0 = session.sql("""
    SELECT * FROM TABLE(
        MODEL_MONITOR_PERFORMANCE_METRIC(
            'ML_DEMO.ML_CHURN.CHURN_MONITOR',
            METRIC_NAME => 'CLASSIFICATION_ACCURACY',
            GRANULARITY => '1 DAY',
            SEGMENTS => '{"SEGMENTS": [{"column": "CONTRACT_TYPE_SEG", "value": "0"}]}'
        )
    )
    """).to_pandas()
    print("Accuracy for month-to-month customers (CONTRACT_TYPE=0):")
    display(seg_accuracy_0)
except Exception as e:
    print(f"Segment metrics not yet available: {e}")

# Query accuracy for CONTRACT_TYPE = '1' (one-year)
try:
    seg_accuracy_1 = session.sql("""
    SELECT * FROM TABLE(
        MODEL_MONITOR_PERFORMANCE_METRIC(
            'ML_DEMO.ML_CHURN.CHURN_MONITOR',
            METRIC_NAME => 'CLASSIFICATION_ACCURACY',
            GRANULARITY => '1 DAY',
            SEGMENTS => '{"SEGMENTS": [{"column": "CONTRACT_TYPE_SEG", "value": "1"}]}'
        )
    )
    """).to_pandas()
    print("\nAccuracy for one-year customers (CONTRACT_TYPE=1):")
    display(seg_accuracy_1)
except Exception as e:
    print(f"Segment metrics not yet available: {e}")

In [ ]:
# Query drift (PSI) per segment -- are certain contract types drifting more?
try:
    seg_psi_0 = session.sql("""
    SELECT * FROM TABLE(
        MODEL_MONITOR_DRIFT_METRIC(
            'ML_DEMO.ML_CHURN.CHURN_MONITOR',
            METRIC_NAME => 'PSI',
            COLUMN_NAME => '"PREDICTED_CHURNED"',
            GRANULARITY => '1 DAY',
            SEGMENTS => '{"SEGMENTS": [{"column": "CONTRACT_TYPE_SEG", "value": "0"}]}'
        )
    )
    """).to_pandas()
    print("PSI for month-to-month customers (CONTRACT_TYPE=0):")
    display(seg_psi_0)
except Exception as e:
    print(f"Segment drift not yet available: {e}")

## 7. Monitor Management

Common operations for managing an existing monitor. These commands are useful for day-to-day operations, debugging, and configuration changes.

### Inspect Monitor Configuration

`DESCRIBE MODEL MONITOR` shows the full configuration including state, columns, refresh interval, and any errors. `SHOW MODEL MONITORS` lists all monitors in a schema.

In [ ]:
# Show monitor details
try:
    desc = session.sql("DESC MODEL MONITOR ML_DEMO.ML_CHURN.CHURN_MONITOR").to_pandas()
    print("Monitor configuration:")
    display(desc)
except Exception:
    # DESC MODEL MONITOR may not be available on all accounts
    desc = session.sql("SHOW MODEL MONITORS LIKE 'CHURN_MONITOR' IN SCHEMA ML_DEMO.ML_CHURN").to_pandas()
    print("Monitor info:")
    display(desc)

In [ ]:
# List all monitors in the schema
monitors = session.sql("SHOW MODEL MONITORS IN SCHEMA ML_DEMO.ML_CHURN").to_pandas()
print("All monitors in ML_DEMO.ML_CHURN:")
monitors

### Suspend and Resume

Suspending a monitor pauses its refresh job -- useful for maintenance or cost control. Resuming restarts it from where it left off. The monitor also auto-suspends after 5 consecutive refresh failures.

In [ ]:
# Suspend the monitor
session.sql("ALTER MODEL MONITOR ML_DEMO.ML_CHURN.CHURN_MONITOR SUSPEND").collect()
print("Monitor suspended")

# Check state via SHOW
state = session.sql("SHOW MODEL MONITORS LIKE 'CHURN_MONITOR' IN SCHEMA ML_DEMO.ML_CHURN").to_pandas()
print(f"State: {state['monitor_state'].iloc[0] if 'monitor_state' in state.columns else 'see output'}")

# Resume the monitor
session.sql("ALTER MODEL MONITOR ML_DEMO.ML_CHURN.CHURN_MONITOR RESUME").collect()
print("Monitor resumed")

### Change Refresh Interval or Warehouse

You can adjust the refresh frequency and compute warehouse without recreating the monitor. Useful for tuning cost vs freshness.

In [ ]:
# Change refresh interval (e.g., from 1 minute to 1 hour for production)
session.sql("""
ALTER MODEL MONITOR ML_DEMO.ML_CHURN.CHURN_MONITOR SET
    REFRESH_INTERVAL = '1 hour'
""").collect()
print("Refresh interval changed to 1 hour")

# Change it back for demo purposes
session.sql("""
ALTER MODEL MONITOR ML_DEMO.ML_CHURN.CHURN_MONITOR SET
    REFRESH_INTERVAL = '1 minute'
""").collect()
print("Refresh interval reset to 1 minute")

### Viewing in Snowsight

You can also view all monitor metrics in the Snowsight UI:

**Navigate to: AI & ML > Models > CHURN_MODEL > Monitors**

The UI shows time-series charts for all configured metrics, with filters for time range, metric type, and segment. This is the easiest way to get an overview of model health without writing SQL.

## 8. Cleanup

Remove all objects created by this notebook. Shared resources from `00_setup.ipynb` (database, schema, warehouse, model, train/test tables) are left intact so other demo notebooks can still run.

The monitor must be suspended before it can be dropped.

In [ ]:
# Suspend then drop the monitor
session.sql("ALTER MODEL MONITOR IF EXISTS ML_DEMO.ML_CHURN.CHURN_MONITOR SUSPEND").collect()
session.sql("DROP MODEL MONITOR IF EXISTS ML_DEMO.ML_CHURN.CHURN_MONITOR").collect()
print("Dropped CHURN_MONITOR")

# Drop prediction and baseline tables
session.sql("DROP TABLE IF EXISTS ML_DEMO.ML_CHURN.CHURN_PREDICTIONS").collect()
session.sql("DROP TABLE IF EXISTS ML_DEMO.ML_CHURN.CHURN_BASELINE").collect()
print("Dropped CHURN_PREDICTIONS and CHURN_BASELINE")

print("\nAll model monitor objects cleaned up.")

In [ ]:
session.close()
print("Session closed.")